# CrossSkinNet — Kaggle training, validation, prediction

বাংলা: GPU চালু করো। Code ZIP, skin dataset ও saved Hugging Face model Add Input দিয়ে attach করো। নিচের cells ক্রমানুসারে চালাও। আগের notebook-এর patches লাগবে না। Accuracy আগে থেকে নিশ্চিত নয়।

Default: a short **image-only smoke experiment**, then change SMOKE_RUN=False for full training. Compare settings on validation only; final test is disabled until you explicitly enable it. Source model files must include config.json, weights, and preprocessor_config.json. No model weights or patient data are included in this code package.
GitHub option: clone https://github.com/mahabubhasanmahin/model.git into /kaggle/working/model before the setup cell; no code-dataset upload is then needed.


In [ ]:
from pathlib import Path
import shutil, zipfile, sys

DEST = Path('/kaggle/working/CrossSkinNet_Kaggle')
clone = Path('/kaggle/working/model/kaggle_release')
if (clone / 'main.py').is_file():
    roots = [clone]
else:
    roots = [p.parent for p in Path('/kaggle/input').rglob('main.py')
             if (p.parent / 'crossskinnet' / 'engine.py').is_file()]
if len(roots) == 1:
    shutil.copytree(roots[0], DEST, dirs_exist_ok=True)
elif len(roots) > 1:
    raise ValueError(f'Multiple code inputs: {roots}. Attach only the intended version.')
else:
    archives = list(Path('/kaggle/input').rglob('CrossSkinNet-Kaggle.zip'))
    if len(archives) != 1:
        raise FileNotFoundError('Attach the CrossSkinNet code dataset first.')
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target = (Path('/kaggle/working') / member.filename).resolve()
            if not target.is_relative_to(Path('/kaggle/working')):
                raise ValueError('Invalid archive path')
        archive.extractall('/kaggle/working')
if not (DEST / 'main.py').exists():
    raise FileNotFoundError(DEST)
%cd /kaggle/working/CrossSkinNet_Kaggle
sys.path.insert(0, str(DEST))


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch, torchvision, transformers
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('transformers:', transformers.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('Enable a Kaggle GPU before full training. CPU is OK for the software smoke test.')


If imports fail after package installation, restart the notebook session and rerun the cells. Keep Kaggle's preinstalled matching torch/torchvision pair; do not independently upgrade one. Online gated models: put HF_TOKEN in Kaggle Secrets and authenticate with the optional cell below. Skip it for local models.

In [ ]:
# Optional online model authentication; no tokens in source/config.
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)


In [ ]:
# Fast offline software validation, unrelated to disease accuracy.
import subprocess, sys
subprocess.run([sys.executable, 'smoke_test.py'], check=True)


## Configuration — edit this cell

`SMOKE_RUN=True`: 1+1 epochs on all selected training data, for a pipeline check (can still take time). `False`: full epoch budgets with early stopping. Set MODE to image_only first; later use gated / metadata_only / naive with the SAME split directory. Flip only one experiment setting at a time. Missing lesion IDs are explicitly excluded and counted; this is lesion grouping, not verified patient grouping.

In [ ]:
import json, yaml, uuid
from pathlib import Path

# Set explicit paths if automatic discovery is ambiguous.
MODEL_PATH = None   # or '/kaggle/input/.../dinov3-vitb16'
LABELS_CSV = None
METADATA_CSV = None
IMAGE_ROOT = None
BACKBONE = 'dinov3'
MODE = 'image_only'  # image_only | gated | metadata_only | naive
AUGMENTATION = False
SMOKE_RUN = True

inputs = Path('/kaggle/input')
def exactly_one(paths, description):
    paths = list(paths)
    if len(paths) != 1:
        raise ValueError(f'{description}: found {paths}; set an explicit path above.')
    return str(paths[0])

if MODEL_PATH is None:
    expected = {'dinov3':'dinov3_vit', 'dinov2':'dinov2', 'efficientnet':'efficientnet', 'swin':'swin'}[BACKBONE]
    candidates = []
    for p in inputs.rglob('config.json'):
        try:
            if json.loads(p.read_text()).get('model_type') == expected:
                candidates.append(p.parent)
        except (ValueError, OSError):
            continue
    MODEL_PATH = exactly_one(candidates, 'Model folder')
if LABELS_CSV is None:
    LABELS_CSV = exactly_one(inputs.rglob('ISIC_2019_Training_GroundTruth.csv'), 'Labels CSV')
if METADATA_CSV is None:
    METADATA_CSV = exactly_one(inputs.rglob('ISIC_2019_Training_Metadata.csv'), 'Metadata CSV')
if IMAGE_ROOT is None:
    IMAGE_ROOT = exactly_one((p for p in inputs.rglob('ISIC_2019_Training_Input') if p.is_dir()), 'Images folder')

config = yaml.safe_load(Path('configs/isic2019.yaml').read_text())
config['model'].update(backbone=BACKBONE, path=MODEL_PATH, mode=MODE, use_fast=BACKBONE=='dinov3')
config['data'].update(labels_csv=LABELS_CSV, metadata_csv=METADATA_CSV, image_root=IMAGE_ROOT)
config['training'].update(device='cuda' if torch.cuda.is_available() else 'cpu', augmentation=AUGMENTATION)
if SMOKE_RUN:
    config['training']['phases'][0]['epochs'] = 1
    config['training']['phases'][1]['epochs'] = 1
config['run']['output_dir'] = f'/kaggle/working/outputs/{MODE}_{uuid.uuid4().hex[:8]}'
CONFIG_PATH = '/kaggle/working/experiment.yaml'
Path(CONFIG_PATH).write_text(yaml.safe_dump(config, sort_keys=False))
print(yaml.safe_dump(config, sort_keys=False))


In [ ]:
from crossskinnet.data import prepare_splits
parts, split_info = prepare_splits(config)
print(json.dumps(split_info['audit'], indent=2))
for name, frame in parts.items():
    print(name, len(frame), frame['_label'].value_counts().to_dict())


In [ ]:
# Train + validation only. Does not evaluate the test split.
from crossskinnet.engine import train
CHECKPOINT = train(config)
print('Best:', CHECKPOINT)
print('Resume file:', CHECKPOINT.parent / 'last.pt')


In [ ]:
from crossskinnet.engine import evaluate
VAL_RESULTS = evaluate(CHECKPOINT, split='val', device=config['training']['device'])
import pandas as pd
history = json.loads((CHECKPOINT.parent / 'history.json').read_text())
display(pd.DataFrame([{'epoch':r['epoch'], 'phase':r['phase'], 'loss':r['train_loss'],
                       'val_accuracy':r['validation']['accuracy'],
                       'val_macro_f1':r['validation']['macro_f1']} for r in history]))
display(pd.DataFrame(VAL_RESULTS['0.0']['per_class']).T)
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename=str(CHECKPOINT.parent / 'val_confusion.png')))


## Controlled comparisons

For a new run, duplicate the configuration, keep `data` and split_seed unchanged, choose a NEW output_dir, and change one setting: augmentation, mode, loss or unfreeze_blocks. Do not rerun a finished run into its old output directory. `python main.py compare RUN_A RUN_B` checks split fingerprints and compares validation metrics only. A four-block schedule is optional; enable it in a separately named run. Metadata-only and naive baselines have different missingness behavior by design; all use the same schema and classes.

Resume: use `last.pt`, with all configuration fields unchanged except output_dir. Keep best.pt, processor/, and last.pt together. Split directory and original image paths must still be available. Changing epochs/LR is a new experiment, not an exact resume. A completed run resumes as a no-op; extend it only via a new experiment design.

In [ ]:
# Example resume after interruption (do not run for an already finished experiment):
# CHECKPOINT = train(config, resume=Path(config['run']['output_dir']) / 'last.pt')


In [ ]:
# Final test: enable only after model/hyperparameters are selected on validation.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    TEST_RESULTS = evaluate(CHECKPOINT, split='test', device=config['training']['device'],
                            stress=True, bootstrap=200)
    display(pd.DataFrame(TEST_RESULTS['0.0']['per_class']).T)
else:
    print('Final test skipped intentionally. Set RUN_FINAL_TEST=True when ready.')


In [ ]:
# One or two new images: this is inference, not dataset accuracy measurement.
from crossskinnet.engine import predict
IMAGE_PATH = ''  # '/kaggle/input/.../patient_image.jpg'
METADATA = {'age_approx':45, 'sex':'male', 'anatom_site_general':'posterior torso'}
# Use {} for all metadata missing. Values must use dataset vocabulary/coding.
if IMAGE_PATH:
    print(json.dumps(predict(CHECKPOINT, IMAGE_PATH, METADATA,
                             config['training']['device']), indent=2))


In [ ]:
# Optional Grad-CAM. SHAP requires requirements-xai.txt and can be slow.
RUN_XAI = False
RUN_SHAP = False
if RUN_XAI and IMAGE_PATH:
    if RUN_SHAP:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-xai.txt'], check=True)
    from crossskinnet.xai import explain
    explain(CHECKPOINT, IMAGE_PATH, METADATA, config['training']['device'],
            CHECKPOINT.parent / 'explanations', with_shap=RUN_SHAP)


In [ ]:
# Save trained artifacts; includes metadata/split CSVs, so keep patient data private.
import shutil
archive = shutil.make_archive('/kaggle/working/crossskin_results', 'zip',
                              root_dir=str(CHECKPOINT.parent))
print(archive)
# Kaggle: Save Version to retain outputs; processes do not survive a session restart.
